In [ ]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:100% !important;}
div.cell.code_cell.rendered{width:100%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:20pt;}
div.text_cell_render.rendered_html{font-size:18pt;}
div.text_cell_render.rendered_html{font-size:15pt; width:90%;}
div.output {font-size:18pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:18pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:18pt;padding:5px;}
table.dataframe{font-size:18px;}
</style>
"""))

# Attention 으로 번역기 만들기
- Attention

## 1. 패키지 import & 하이퍼파라미터
- 하이퍼파라미터 : 모델의 정확도 및 학습속도에 영향을 미치는 변수

In [ ]:
import numpy as np
import pandas as pd
from time import time

from tensorflow.keras.layers import Input, LSTM, Dense, Attention, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

# 하이퍼파라미터
MY_HIDDEN = 128
MY_EPOCH = 500

## 2. 번역 데이터 불러오기

In [ ]:
raw = pd.read_csv('data/translate.csv', header=None)
eng_kor = raw.values.tolist() # 데이터프레임을 list로 변환

## 3. 영어알파벳과 한글문자 리스트 만들기

In [ ]:
# 영어 알파벳 리스트(e_alpha)
e_alpha = [c for c in 'SEPabcdefghijklmnopqrstuvwxyz']

# 한글 문자 리스트(k_ch, k_alpha)
korean = ''.join([data[1] for data in eng_kor])
k_ch = list(set([ch for ch in korean]))
k_ch.sort()
# print(k_ch)

k_alpha = pd.read_csv('data/korean.csv', header=None)[0].tolist()
k_alpha == k_ch # 순서와 내용이 모두 같음

In [ ]:
alpha = e_alpha + k_alpha
alpha_total_size = len(alpha)

## 4. 문자당 num를 갖는 dict 만들기

In [ ]:
char_to_num = {c:i for i, c in enumerate(alpha)}
print(char_to_num)

In [ ]:
data = eng_kor[0]
print(data)
print(char_to_num['c'], char_to_num['o'], char_to_num['l'], char_to_num['d'])
print('인코더 입력(원핫인코딩전) :',[char_to_num[c] for c in data[0]])
print('디코더 입력(원핫인코딩전) :', [char_to_num[c] for c in 'S'+data[1]])
print('디코더 출력(원핫인코딩X) :', [char_to_num[c] for c in data[1]+'E'])

## 5. 인코더 입력, 디코더 입력, 디코더 출력
- 인코더 입력데이터 : 영어알파벳 -> 숫자 -> 원핫인코딩(110,4,171)
- 디코더 입력데이터 : 'S'+한글문자 -> 숫자 -> 원핫인코딩(110,3,171)
- 디코더 타겟데이터 : 한글문자+'E' -> 숫자 shape이 (110,3)인 list 

In [ ]:
def encoding(eng_kor=eng_kor):
    enc_in = [] # 인코더 입력
    dec_in = [] # 디코더 입력
    dec_out = [] # 디코더 출력(타겟)
    for data in eng_kor:
        # 인코더 입력데이터(영어알파벳 -> 숫자 -> 원핫인코딩)
        eng = [char_to_num[c] for c in data[0]]
        eng_one = np.eye(alpha_total_size)[eng]
        # print('영어 :', eng, eng_one)
        enc_in.append(eng_one) # eng_one의 shape : (4,171)
        
        # 디코더 입력데이터("S한글" -> 숫자 -> 원핫인코딩)
        kor = [char_to_num[c] for c in "S"+data[1]]
        #kor_one = to_categorical(kor, num_classes=alpha_total_size)
        kor_one = np.eye(alpha_total_size)[kor] # kor_one의 shape : (3, 171)
        # print('한글 :', kor, kor_one)
        dec_in.append(kor_one)
        
        # 디코더 출력데이터("한글E" -> 숫자)
        kor = [char_to_num[c] for c in data[1]+"E"]
        # print(kor)
        dec_out.append(kor)
    return enc_in, dec_in, dec_out

## 6. 전체 입력데이터, 타겟데이터 준비

In [ ]:
x_enc, x_dec, y_dec = encoding(eng_kor)
X_enc = np.array(x_enc)
X_dec = np.array(x_dec)
# Y_dec = np.array(y_dec).reshape(-1, 3, 1) # 차원 추가
Y_dec = np.expand_dims(y_dec, axis=-1)
# Y_dec = np.array(y_dec)[..., np.newaxis]
# Y_dec = np.array(y_dec)[:,:,None]

X_enc.shape, X_dec.shape, Y_dec.shape

## 7. 모델 구현(Seq2Seq)

In [ ]:
# 인코더 LSTM
ENC_IN = Input(shape=(4, alpha_total_size)) # alpha_total_size:171

_, state_h, state_c = LSTM(units=MY_HIDDEN, # MY_HIDDEN:128
                           return_state=True, # return_state=True h값과 c 받기
                           # return_sequences=False # LSTM윗 출력 안 받음
                          )(ENC_IN) 

# 인코더와 디코더 연결 고리
LINK = [state_h, state_c]

# 디코더 LSTM
DEC_IN = Input(shape=(3, alpha_total_size))
DEC_MID = LSTM(units=MY_HIDDEN, # 128
            #    return_state=False,
               return_sequences=True, # 윗 출력 받음
               )(DEC_IN,
                 initial_state=LINK)
               
# 최종 출력층
DEC_OUT = Dense(units=alpha_total_size,
                activation='softmax')(DEC_MID)
# 모델
model = Model(inputs=[ENC_IN, DEC_IN],
              outputs=DEC_OUT)
model.summary()

## 7. 모델 구현(Attention)

In [ ]:
# 인코더 입력
ENC_IN = Input(shape=(4, alpha_total_size)) # alpha_total_size:171

# 인코드 LSTM : 모든 LSTM 스탭 출력
ENC_OUT, state_h, state_c = LSTM(units=MY_HIDDEN,
     return_sequences=True, # LSTM 모든 스탭의 윗 출력
     return_state=True, # hat값과 c값 받기
     )(ENC_IN)

# 디코더 입력
DEC_IN = Input(shape=(3, alpha_total_size))

# 디코더 LSTM
DEC_LSTM_OUT, _, _ = LSTM(units=MY_HIDDEN, # 128
                        return_sequences=True,
                        return_state=True)(DEC_IN,
                                           initial_state=[state_h,
                                                          state_c])
# Attention vector(매커니즘 정의)
CONTEXT_VECTOR = Attention()([DEC_LSTM_OUT,ENC_OUT])

# 컨텍스트벡터와 디코더LSTM 결과를 concat
CONTEXT_AND_LSTM_OUT = Concatenate()([CONTEXT_VECTOR, DEC_LSTM_OUT])

#출력
OUT = Dense(units=alpha_total_size,
            activation='softmax')(CONTEXT_AND_LSTM_OUT)

# 모델 정의
model = Model(inputs=[ENC_IN, DEC_IN], outputs=OUT)
model.summary()

## 8. 학습과정 설정 & 학습

In [ ]:
model.compile(loss='sparse_categorical_crossentropy',
              optimizer='rmsprop',
              metrics=['accuracy'], # loss만 로그 출력
              )
begin = time()
model.fit([X_enc, X_dec], Y_dec,
          epochs=MY_EPOCH,
          verbose=1)
end = time()
print('학습시간 :', end-begin)

## 9. 모델사용

In [ ]:
# 쉬운문제
easy_test = [['cold', 'PP'],
             ['fact', 'PP'],
             ['love', 'PP'],
             ['luck', 'PP'],
             ['milk', 'PP']]
enc_in, dec_in, _ = encoding(easy_test)
enc_in = np.array(enc_in)
dec_in = np.array(dec_in)
enc_in.shape, dec_in.shape

In [ ]:
# 위의 문제 예측하기
pred = model.predict([enc_in, dec_in])
pred.argmax(axis=-1)

In [ ]:
# cold => 감기([32, 46])

In [ ]:
for test, yhat in zip(easy_test, pred):
    # print(test[0], yhat.argmax(axis=-1))
    eng = test[0]
    hat = np.argmax(yhat, axis=-1)
    kor = ''.join([alpha[h] for h in hat[:-1]])
    print("{} => {}".format(eng, kor))

In [ ]:
# 어려운 문제
hard_test = [['leve', 'PP'],
             ['lvoe', 'PP'],
             ['love', 'PP'],
             ['lovo', 'PP'],
             ['eovl', 'PP']]
enc_in, dec_in, _ = encoding(hard_test)
enc_in = np.array(enc_in)
dec_in = np.array(dec_in)
pred = model.predict([enc_in, dec_in]).argmax(axis=-1)
[''.join([alpha[h] for h in hat[:-1]]) for hat in pred]

In [ ]:
for test, yhat in zip(hard_test, pred):
    eng = test[0]
    kor = ''.join([alpha[h] for h in yhat[:-1]])
    print("{} => {}({})".format(eng, kor, yhat[:-1]))